# [2부] 16. 공고 분류와 검색: JEV(결정 모델)와 EmbeddingGemma 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/rebom-lec/blob/main/labs/p2_16_notice_jev_eg2.ipynb)

> 🧭 **스토리**: RE:봄 수집기는 매일 구조동물 공고를 수백 건 받는다. 사람이 공고마다 성격 태그를 달고 사진이 쓸 만한지 볼 수는 없다.
> 이 노트북은 파이프라인의 두 단계를 직접 붙여 본다.
> ① **분류**: OpenRouter의 결정 모델 **JEV**에 공고의 글과 사진을 **한 요청**으로 보내고, 태그와 사진 검사 결과를 **확률**로 받는다.
> ② **검색**: **EmbeddingGemma 2**로 글과 사진을 벡터 하나로 만들고 '갈색 진돗개, 순한 아이' 같은 말로 찾는다.
>
> ⏱ **25분** · 🖥 CPU로 충분 (GPU가 있으면 §5가 빠름) · 🔑 OpenRouter 키 (수업에서 나눠 준 키)

공고 8건은 수업용으로 지어낸 것이다. 사진은 위키미디어 공용의 자유 이용 사진이고 출처는 `data/notices/ATTRIBUTION.md`에 있다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️

In [ ]:
%pip -q install -U sentence-transformers transformers

In [ ]:
# ⚙️ 실습 데이터 로더: 깃허브(rebom-lec/data)에서 받아 온다. 업로드할 필요 없음.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/rebom-lec/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'notices/photos/n01.jpg') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def set_korean_font():
    """표 · 그림의 한글 폰트. 실패해도 진행한다."""
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    try:
        dst = Path(fetch("BMDOHYEON.ttf"))
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패, 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ OpenRouter 키 입력: 화면에 표시되지 않는다.
# 규칙: 키를 코드에 적지 않는다 · 출력하지 않는다 · 수업이 끝나면 키는 폐기된다.
# 키 확인은 GET /api/v1/key(키 정보 조회)로 한다. 모델을 부르지 않으므로 비용이 없다.
import requests
from getpass import getpass
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API 키: ").strip()
HEAD = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}
r = requests.get("https://openrouter.ai/api/v1/key", headers=HEAD, timeout=20)
print("키 확인:", "OK" if r.status_code == 200 else f"실패 (HTTP {r.status_code}) — 키를 다시 확인하세요")

## 1. 공고 읽기와 개인정보 처리 🔵
공공데이터 원본에는 담당자 이름 · 연락처 필드가 있고, 특징 문구(`specialMark`)에 전화번호가 섞여 오기도 한다.
**모델에 보내기 전에** 필드를 빼고 번호를 가린다(2부 B 블록 전처리). 정규식은 앞뒤에 숫자가 붙지 않을 때만 잡도록 경계를 건다.

In [ ]:
import re, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

PERSONAL_FIELDS = {"chargeNm", "officetel", "careOwnerNm"}          # AI 입력 · 저장소 어디에도 넣지 않는다
PHONE_RE = re.compile(r"(?<!\d)(0\d{1,2})[-.)\s]?(\d{3,4})[-.\s]?(\d{4})(?!\d)")

def clean(n: dict) -> dict:
    n = {k: v for k, v in n.items() if k not in PERSONAL_FIELDS}
    n["specialMark"] = PHONE_RE.sub("[전화번호]", n.get("specialMark") or "")
    return n

raw = json.load(open(fetch("notices/notices.json"), encoding="utf-8"))
NOTICES = [clean(n) for n in raw]
photos = {n["desertionNo"]: Image.open(fetch("notices/" + n["popfile"])).convert("RGB") for n in NOTICES}

print("뺀 필드:", sorted({k for n in raw for k in n} & PERSONAL_FIELDS))
display(pd.DataFrame(NOTICES)[["desertionNo", "upKindNm", "kindNm", "colorCd", "age", "specialMark"]].set_index("desertionNo"))
fig, axes = plt.subplots(1, len(NOTICES), figsize=(2.2 * len(NOTICES), 2.4))
for ax, n in zip(axes, NOTICES):
    ax.imshow(photos[n["desertionNo"]]); ax.axis("off")
    ax.set_title(f"{n['desertionNo'][-2:]} [{n['upKindNm']}] {n['kindNm']}", fontsize=8)
plt.tight_layout(); plt.show()

💡 **해석**: 04번 문구의 휴대전화 번호가 `[전화번호]`로 바뀌었다. 08번은 글에는 **고양이**로 적혀 있는데 사진은 05번과 같은 **개** 사진이다.
실제 공고에서도 사진을 잘못 올리는 일이 있다. 사람이 하나씩 보지 않고 이런 공고를 골라낼 수 있을지 §3에서 본다.

## 2. 글만 보내기: 질문 세 종류를 한 요청에 🔵
JEV는 글을 생성하지 않는다. `state`(판단할 자료)와 `questions`(질문 목록)를 받아 질문마다 **정해진 모양의 답과 확률**을 돌려준다.

| 타입 | 묻는 것 | 돌아오는 값 |
|---|---|---|
| `noul` | 예 · 아니오 | `noul`: '예'일 확률 (0~1) |
| `choice` | 보기 중 하나 | `choice` · 보기별 `probabilities` · `confidence` |
| `score` | 순서가 있는 단계 중 어디쯤 | `score`: 확률 가중 평균 위치 (0 = 첫 단계) |

질문은 한 요청 안에서 **서로 독립적으로** 답한다. 앞 질문의 답을 뒤 질문이 보지 못한다. 출력 토큰은 과금되지 않고 입력 토큰만 과금된다.

In [ ]:
import time
URL = "https://openrouter.ai/api/alpha/decisions"
MODEL = "typesafe/jev-1.13"      # 버전을 고정한다(임계값은 버전마다 다시 정한다). 최신은 ~typesafe/jev-latest

def decide(state, questions, model=MODEL):
    """결정 요청 1회 → (answers, usage, 초). 실패하면 상태 코드만 출력하고 (None, None, 초)."""
    t0 = time.time()
    try:
        r = requests.post(URL, headers={**HEAD, "Content-Type": "application/json"},
                          json={"model": model, "state": state, "questions": questions}, timeout=120)
    except requests.RequestException as e:
        print("요청 실패:", type(e).__name__); return None, None, time.time() - t0
    sec = time.time() - t0
    if r.status_code != 200:
        print(f"HTTP {r.status_code}: 키 · 잔액 · 요청 형식을 확인하세요"); return None, None, sec
    body = r.json()
    return body["answers"], body.get("usage", {}), sec

TEXT_Q = {
    "health_issue": {"type": "noul",
        "instructions": "특징 문구에 부상, 질병, 피부 이상 같은 건강 문제가 적혀 있는가?",
        "criteria": {"true": "다치거나 아픈 곳, 증상이 적혀 있다.",
                     "false": "건강 문제에 대한 언급이 없다."}},
    "temperament": {"type": "choice",
        "instructions": "특징 문구로 보아 이 동물의 성격은 무엇에 가장 가까운가?",
        "criteria": {"온순": "사람을 잘 따르거나 순하다고 적혀 있다.",
                     "활발": "활발하거나 에너지가 많다고 적혀 있다.",
                     "겁 많음": "겁이 많거나, 경계하거나, 숨는다고 적혀 있다.",
                     "공격성": "물거나 공격한다고 적혀 있다.",
                     "정보 없음": "성격에 대한 언급이 없다."}},
    "info_level": {"type": "score",
        "instructions": "이 공고 글이 입양 희망자에게 주는 정보는 어느 정도인가?",
        "criteria": ["품종 · 색 · 나이 같은 기본 정보뿐이다",
                     "성격과 건강 중 하나가 적혀 있다",
                     "성격과 건강이 모두 구체적으로 적혀 있다"]},
}

def notice_state(n):
    """공고 한 건 → state(딕셔너리). 판단에 필요한 필드만 보낸다."""
    return {"종": n["upKindNm"], "품종": n["kindNm"], "털색": n["colorCd"], "나이": n["age"], "특징": n["specialMark"] or "(없음)"}

rows = []
for n in NOTICES:
    ans, use, sec = decide(notice_state(n), TEXT_Q)
    if ans is None: continue
    rows.append({"공고": n["desertionNo"][-2:], "특징": n["specialMark"][:24],
                 "건강 문제(noul)": round(ans["health_issue"]["noul"], 2),
                 "성격(choice)": ans["temperament"]["choice"], "확신도": round(ans["temperament"]["confidence"], 2),
                 "정보량(score 0~2)": round(ans["info_level"]["score"], 2),
                 "입력 토큰": use.get("input_tokens"), "비용($)": use.get("cost"), "초": round(sec, 2)})
TEXT_RESULT = pd.DataFrame(rows).set_index("공고") if rows else pd.DataFrame()   # 호출이 모두 실패하면 빈 표
display(TEXT_RESULT)
if rows:
    print(f"8건 비용 합 ${TEXT_RESULT['비용($)'].sum():.6f} · 평균 지연 {TEXT_RESULT['초'].mean():.2f}초")

💡 **해석**: 답이 문자열이 아니라 **숫자와 정해진 보기**라서 파싱할 것이 없다. `noul` 0.5 근처는 '반쯤 아프다'는 뜻이 아니라 **예와 아니오가 비슷하게 가능하다**는 뜻이다.
`score`는 단계 번호(0, 1, 2)의 확률 가중 평균이라 1.6처럼 소수가 나온다. 06번처럼 특징 문구가 빈 공고는 정보량 점수가 낮게 나와야 한다.

## 3. 글과 사진을 한 요청으로 🔵
사진은 `state`를 **배열**로 만들고 그 안에 `image_url` 부분으로 넣는다(PNG · JPEG, base64 data URL). 원격 http 주소는 가져오지 않는다.
한 변이 256~1,568px이어야 하고 EXIF 회전은 적용되지 않아서, `image_part()`가 미리 돌리고 크기를 맞춘다.
§2의 글 질문 세 개에 사진 질문 세 개를 더해 **한 요청에 질문 여섯 개**를 보낸다.

In [ ]:
import base64, io

from PIL import ImageOps

def image_part(img, max_side=768, min_side=256):
    """PIL 이미지 → state에 넣는 image_url 부분 (JPEG base64).
    JEV 제한: 한 변 256~1,568px, EXIF 회전은 적용하지 않음 → 여기서 돌리고 크기를 맞춘다."""
    im = ImageOps.exif_transpose(img).convert("RGB"); im.thumbnail((max_side, max_side))
    if min(im.size) < min_side:                       # 너무 작으면 짧은 변이 256px이 되게 키운다
        r = min_side / min(im.size); im = im.resize((round(im.size[0] * r), round(im.size[1] * r)))
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=85)
    return {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()}}

PHOTO_Q = {
    "photo_usable": {"type": "noul",
        "instructions": "사진에서 동물의 얼굴과 몸이 또렷하게 보여 입양 공고 사진으로 쓸 수 있는가?",
        "criteria": {"true": "동물이 크게, 초점이 맞게 보인다.",
                     "false": "동물이 작거나, 가려지거나, 흐리거나, 일부만 보인다."}},
    "species_match": {"type": "noul",
        "instructions": "사진 속 동물의 종(개 또는 고양이)이 공고 글에 적힌 종과 같은가?",
        "criteria": {"true": "사진의 동물과 글의 종이 같다.",
                     "false": "사진의 동물과 글의 종이 다르다."}},
    "coat": {"type": "choice",
        "instructions": "사진 속 동물의 주된 털색은 무엇인가?",
        "criteria": {"흰색 · 크림": "흰색이나 크림색이 대부분이다.", "갈색 · 황색": "갈색이나 누런색이 대부분이다.",
                     "검정": "검은색이 대부분이다.", "회색": "회색이 대부분이다.", "여러 색 · 얼룩": "두 가지 이상 색이 섞여 있다."}},
}

def notice_text(n):
    s = notice_state(n)
    return "공고 글: " + " · ".join(f"{k} {v}" for k, v in s.items())

rows = []
for n in NOTICES:
    state = [notice_text(n), "공고 사진:", image_part(photos[n["desertionNo"]])]
    ans, use, sec = decide(state, {**TEXT_Q, **PHOTO_Q})
    if ans is None: continue
    rows.append({"공고": n["desertionNo"][-2:], "글의 종": n["upKindNm"],
                 "사진 사용 가능": round(ans["photo_usable"]["noul"], 2), "종 일치": round(ans["species_match"]["noul"], 2),
                 "털색": ans["coat"]["choice"], "털색 확신도": round(ans["coat"]["confidence"], 2),
                 "건강 문제": round(ans["health_issue"]["noul"], 2), "성격": ans["temperament"]["choice"],
                 "성격 확신도": round(ans["temperament"]["confidence"], 2), "정보량": round(ans["info_level"]["score"], 2),
                 "입력 토큰": use.get("input_tokens"), "비용($)": use.get("cost"), "초": round(sec, 2)})
MM_RESULT = pd.DataFrame(rows).set_index("공고") if rows else pd.DataFrame()     # 호출이 모두 실패하면 빈 표(§4 · §5는 건너뛰는 길로 간다)
display(MM_RESULT)
if rows:
    print(f"8건 비용 합 ${MM_RESULT['비용($)'].sum():.6f} · 평균 지연 {MM_RESULT['초'].mean():.2f}초 (글만 보낼 때와 비교)")

In [ ]:
# 🔵 §3 결과를 사진과 나란히: 확률만 보고는 '사진 사용 가능' · '종 일치'가 맞는지 판단하기 어렵다
import io, base64
from IPython.display import HTML, display

def pil_to_thumb_html(img, size=(90, 90)):
    """PIL 이미지 -> HTML <img> 썸네일"""
    im = img.copy()
    im.thumbnail(size)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<img src="data:image/jpeg;base64,{b64}" style="height:{size[1]}px; border:1px solid #ccc;">'

# MM_RESULT에 공고번호(01~08)가 index로 들어 있으므로, 원래 desertionNo와 매핑
idx_to_full_id = {n["desertionNo"][-2:]: n["desertionNo"] for n in NOTICES}

if len(MM_RESULT):
    MM_VIEW = MM_RESULT.copy()
    MM_VIEW.insert(0, "사진", [pil_to_thumb_html(photos[idx_to_full_id[idx]]) for idx in MM_VIEW.index])
    wanted_cols = ["사진", "글의 종", "사진 사용 가능", "종 일치", "털색", "털색 확신도",
                   "건강 문제", "성격", "성격 확신도", "정보량", "입력 토큰", "비용($)", "초"]
    display(HTML(MM_VIEW[wanted_cols].to_html(escape=False)))
else:
    print("(건너뜀) §3 결과가 없습니다.")

In [ ]:
# 🔵 같은 결과를 공고 카드로: 사진 한 장 옆에 글의 종과 JEV 판단 다섯 개
cards = []
for idx, row in MM_RESULT.iterrows():
    img_html = pil_to_thumb_html(photos[idx_to_full_id[idx]], size=(160, 160))
    cards.append(f"""
    <div style="width:240px; border:1px solid #ddd; border-radius:8px; padding:10px; margin:8px;
                display:inline-block; vertical-align:top; text-align:center;">
        <div style="font-weight:bold; margin-bottom:6px;">공고 {idx}</div>
        <div>{img_html}</div>
        <div style="margin-top:8px; text-align:left; font-size:13px;">
            <div><b>글의 종</b>: {row['글의 종']}</div>
            <div><b>종 일치</b>: {row['종 일치']}</div>
            <div><b>사진 사용 가능</b>: {row['사진 사용 가능']}</div>
            <div><b>털색</b>: {row['털색']}</div>
            <div><b>성격</b>: {row['성격']}</div>
            <div><b>건강 문제</b>: {row['건강 문제']}</div>
        </div>
    </div>
    """)
if cards:
    display(HTML("".join(cards)))
else:
    print("(건너뜀) §3 결과가 없습니다.")

💡 **해석**: 08번(글은 고양이, 사진은 개)의 **종 일치** 확률이 낮게 나오는지 본다. 06번(철창 너머로 멀리 누운 개)과 07번(코만 크게 찍힌 사진)은 **사진 사용 가능** 확률을 본다.
같은 글 질문(건강 · 성격 · 정보량)의 값이 §2와 조금 달라질 수 있다. OpenRouter 문서에 따르면 **이미지가 들어간 요청은 JEV의 긴 문맥 모드로 처리되고, 이 모드는 더 느리고 덜 정확하다.**
그래서 실제 파이프라인에서는 글 질문은 글만 보내고, 사진 질문만 사진과 함께 보내는 방법도 있다. 비용 · 지연 · 정확도를 표로 비교해 고른다.

## 4. 확률로 나누기: 자동 반영과 사람 확인 🔵
확률이 있으니 "애매하면 사람에게"를 코드로 쓸 수 있다. 아래 임계값은 **예시**다. 실제 값은 사람이 정답을 붙인 공고 수십~수백 건으로 정하고,
틀렸을 때 어느 쪽이 더 비싼지(잘못된 사진이 검색에 뜨는 것 vs 멀쩡한 공고가 확인 대기에 쌓이는 것)를 보고 고른다.

In [ ]:
TH = {"species_match": 0.8,      # 엄격하게: 다른 동물 사진이 공고에 붙으면 신뢰가 깎인다
      "photo_usable": 0.5,
      "temperament_conf": 0.4,  # 느슨하게: 성격 태그는 틀려도 피해가 작다
      "health": 0.7}            # 건강 확인 태그를 붙이는 기준

def route(row):
    why = []
    if row["종 일치"] < TH["species_match"]: why.append("사진과 글의 종이 다름")
    if row["사진 사용 가능"] < TH["photo_usable"]: why.append("사진 품질")
    if row["성격 확신도"] < TH["temperament_conf"]: why.append("성격 태그 애매")
    tags = ([row["성격"]] if row["성격 확신도"] >= TH["temperament_conf"] and row["성격"] != "정보 없음" else []) \
         + (["건강 확인 필요"] if row["건강 문제"] >= TH["health"] else [])
    return pd.Series({"처리": "사람 확인" if why else "자동 반영", "이유": ", ".join(why), "붙일 태그": ", ".join(tags),
                      "사진 색인": row["종 일치"] >= TH["species_match"] and row["사진 사용 가능"] >= TH["photo_usable"]})

if len(MM_RESULT):
    ROUTED = MM_RESULT.apply(route, axis=1)
    display(ROUTED)
    per_1000 = MM_RESULT["비용($)"].mean() * 1000
    print(f"공고 1,000건 예상 비용 ${per_1000:.3f} (이번 8건의 usage.cost 평균 × 1,000) · 하루 600건이면 ${per_1000 * 0.6:.3f}")

💡 **해석**: `confidence`는 보기들 사이에서 확률이 얼마나 한쪽으로 몰렸는지를 말한다. **이 판단이 안전한지**를 말하지는 않는다(공식 문서의 주의).
같은 질문이라도 모델 버전이 바뀌면 확률 분포가 달라진다. 그래서 모델 이름을 `typesafe/jev-1.13`처럼 고정하고, 바꿀 때 임계값을 다시 정한다.

## 5. EmbeddingGemma 2로 글과 사진을 함께 색인 🔵
10번 노트북에서 본 EG2는 글과 사진을 한 공간에 넣는다. 글 사이에 `<|image|>` 자리를 두면 **글과 사진을 섞은 입력도 벡터 하나**가 된다.
§4 결과로 색인 방법을 정한다. 종이 맞지 않는 공고는 확인이 끝날 때까지 색인에서 빼고, 사진 품질이 낮은 공고는 글만, 나머지는 글 + 사진을 섞는다. 비교용으로 글만, 사진만 만든 색인도 함께 만든다.

In [ ]:
import time, torch, numpy as np
from sentence_transformers import SentenceTransformer
HAS_GPU = torch.cuda.is_available()
EG_DTYPE = torch.bfloat16 if (HAS_GPU and torch.cuda.is_bf16_supported()) else torch.float32   # float16은 쓰지 않는다
t0 = time.time()
eg = SentenceTransformer("google/embeddinggemma-2", device="cuda" if HAS_GPU else "cpu",
                         model_kwargs={"dtype": EG_DTYPE}, config_kwargs={"audio_config": None})   # 음성 인코더는 뺀다
print(f"로드 {time.time() - t0:.1f}초 · {EG_DTYPE} · 차원 {eg.get_embedding_dimension()}")

In [ ]:
def doc_text(n):
    return f"{n['upKindNm']} {n['kindNm']}, {n['colorCd']}, {n['age'][:4]}년생. {n['specialMark']}".strip()

# §4 결과로 색인 방법을 정한다: 종이 다르면 확인이 끝날 때까지 색인에 넣지 않고, 사진 품질이 낮으면 글만 넣는다
if len(MM_RESULT):
    HOLD = {k for k, r in ROUTED.iterrows() if "종이 다름" in r["이유"]}
    USE_PHOTO = ROUTED["사진 색인"].to_dict()
else:                                  # JEV 셀을 건너뛰었으면 모두 사진을 섞는다
    HOLD, USE_PHOTO = set(), {n["desertionNo"][-2:]: True for n in NOTICES}

def build(mode):
    keys, vecs = [], []
    for n in NOTICES:
        k = n["desertionNo"][-2:]
        if mode == "섞음(검사 반영)" and k in HOLD:
            continue                   # 확인 대기: 검색에 내보내지 않는다
        if mode == "사진만" or (mode == "섞음(검사 반영)" and USE_PHOTO.get(k)):
            if mode == "사진만":
                v = eg.encode({"image": photos[n["desertionNo"]]}, normalize_embeddings=True)
            else:                      # 글 + 사진을 섞어 벡터 하나
                v = eg.encode({"text": f"{n['upKindNm']} {n['kindNm']}, {n['colorCd']}. <|image|> {n['specialMark']}",
                               "image": [photos[n["desertionNo"]]]}, normalize_embeddings=True)
        else:                          # 글만 (사진을 믿을 수 없거나 '글만' 색인)
            v = eg.encode(doc_text(n), prompt_name="Document", normalize_embeddings=True)
        keys.append(k); vecs.append(v)
    return keys, np.stack(vecs)

t0 = time.time()
INDEX = {m: build(m) for m in ("글만", "사진만", "섞음(검사 반영)")}
print(f"색인 3종 · {time.time() - t0:.1f}초 · 확인 대기로 뺀 공고: {sorted(HOLD) or '없음'} · 사진을 섞은 공고: {sorted(k for k, v in USE_PHOTO.items() if v and k not in HOLD)}")

In [ ]:
QUERIES = ["갈색 진돗개, 순한 아이", "하얀 소형견", "고양이", "겁 많은 개"]
Q = eg.encode(QUERIES, prompt_name="SearchQuery", normalize_embeddings=True)
by_key = {n["desertionNo"][-2:]: n for n in NOTICES}
rows = []
for name, (keys, M) in INDEX.items():
    S = Q @ M.T
    for q, s in zip(QUERIES, S):
        top = np.argsort(-s)[:3]
        rows.append({"색인": name, "질문": q, **{f"{r + 1}등": f"{keys[j]} ({s[j]:.3f})" for r, j in enumerate(top)}})
display(pd.DataFrame(rows).set_index(["질문", "색인"]).sort_index())

q = "고양이"; keys, M = INDEX["섞음(검사 반영)"]
s = eg.encode([q], prompt_name="SearchQuery", normalize_embeddings=True) @ M.T
fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
for ax, j in zip(axes, np.argsort(-s[0])[:3]):
    n = by_key[keys[j]]; ax.imshow(photos[n["desertionNo"]]); ax.axis("off")
    ax.set_title(f"{keys[j]} [{n['upKindNm']}] {s[0][j]:.3f}", fontsize=9)
plt.suptitle(f"'{q}' 상위 3건 (섞음 색인)", fontsize=10); plt.tight_layout(); plt.show()

💡 **해석**: '고양이'로 찾을 때 **글만** 색인은 08번(글은 고양이, 사진은 개)을 1등으로 올린다. **사진만** 색인은 글에 적힌 성격 · 건강을 모른다.
**섞음(검사 반영)** 색인은 종이 맞지 않는 08번을 확인이 끝날 때까지 빼고, 사진 품질이 낮은 공고는 글만 넣는다. 08번을 글만으로 넣어 두면 '고양이'에서 여전히 1등이 되므로, 맞지 않는 공고는 색인에서 빼는 쪽이 안전하다.
점수는 색인마다 분포가 다르므로(10번 노트북의 모달리티 격차) **색인끼리 점수를 직접 비교하지 않는다.**

## 6. 🤔 생각해보기
1) 하루 600건이 들어온다면 JEV 분류 비용과 지연은 얼마일까? §2(글만)와 §3(글 + 사진) 중 어떤 조합으로 보낼까?
2) 임계값을 정하려면 정답 붙인 공고가 몇 건쯤 필요할까? 누가, 어떤 기준으로 붙일까?
3) EG2 색인을 만든 뒤 JEV 임계값을 바꾸면 무엇을 다시 해야 할까?

<details><summary><b>🤔 생각해보기: 해설 보기(클릭)</b></summary>

1) §4가 출력한 '공고 1,000건 예상 비용'에 0.6을 곱한다. 출력 토큰은 무료이고 입력 토큰만 과금되므로 사진 크기(`max_side`)가 비용을 좌우한다. 글 질문은 글만 보내면 빠르고, 이미지 요청의 긴 문맥 모드를 피할 수 있다.
2) 공식 가이드는 임계값을 감으로 정하지 말고 정답 붙인 표본으로 정하라고 한다. 질문마다 예 · 아니오가 섞인 수십 건부터 시작하고, 틀린 사례를 모아 늘린다. 기준(예: '사진 사용 가능'의 정의)을 문서로 먼저 쓴다.
3) JEV 점수(확률)를 공고별로 저장해 두었다면 JEV를 다시 부르지 않고 새 임계값으로 다시 판정만 하면 된다. 임계값만 바뀌었으니 EG2 임베딩도 대부분 그대로 쓴다.
다만 이 노트북의 '섞음' 색인은 판정 결과에 따라 임베딩 입력이 달라진다(사진을 믿으면 글 + 사진, 아니면 글만). 그래서 판정이 바뀐 공고만 다시 임베딩한다. 공고마다 '글만' 벡터와 '글 + 사진' 벡터를 둘 다 만들어 저장해 두면 다시 임베딩하지 않고 색인에 넣을 벡터만 바꾸면 된다. 확인 대기로 빠지는 공고는 색인에서 빼기만 한다.
설계로 보면 JEV의 품질 판단과 EG2의 임베딩 생성을 나눠 두어야 임계값이나 정책을 바꿀 때마다 불필요한 임베딩 계산이 생기지 않는다. 벡터와 함께 JEV 모델 버전 · 임계값 · EG2 모델 이름을 저장해 두면 무엇을 다시 만들지 알 수 있다.
</details>